# Generation eval on a Colab GPU

Runs `hybrid_rag.eval_generation` with `qwen2.5:7b` served by Ollama on the Colab GPU.
Same code as local runs; only the hardware differs (record it in the README).

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then run the cells in order.
Results are written to Google Drive, so if the session disconnects you can re-run all cells
and the eval resumes where it stopped.

In [ ]:
# 1. Confirm a GPU is attached (should list a Tesla T4 or similar)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Mount Google Drive: results go here so they survive a disconnect
from google.colab import drive

drive.mount("/content/drive")
RESULTS = "/content/drive/MyDrive/hybrid-rag-results/run3"  # new folder per code version
!mkdir -p "$RESULTS"

In [ ]:
# 3. Upload hybrid-rag-src.zip (made on your Mac) and unpack it
from google.colab import files

uploaded = files.upload()
assert "hybrid-rag-src.zip" in uploaded, f"Expected hybrid-rag-src.zip, got {list(uploaded)}"
!rm -rf /content/hybrid-rag && mkdir -p /content/hybrid-rag
!unzip -q -o hybrid-rag-src.zip -d /content/hybrid-rag
%cd /content/hybrid-rag
!ls

In [ ]:
# 4. Install Ollama, start the server in the background, download the model (~4.7 GB)
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess
import time
import urllib.request

subprocess.Popen(
    ["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT
)
for _ in range(30):
    try:
        print(urllib.request.urlopen("http://localhost:11434/api/version").read().decode())
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start; see /content/ollama.log")
!ollama pull qwen2.5:7b

In [ ]:
# 5. Install, fetch corpus, build both indexes (CPU, a few minutes each)
!pip install -q -e .
!bash scripts/fetch_corpus.sh
!python -m hybrid_rag.build_index --strategy header
!python -m hybrid_rag.build_index --strategy fixed

In [ ]:
# 6. Run the generation eval. Re-running this cell resumes (finished questions are skipped).
!python -m hybrid_rag.eval_generation --strategy header --mode hybrid --out-dir "$RESULTS"

In [ ]:
# 7. Check the GPU was really used: Ollama should report the model on the GPU (PROCESSOR = 100% GPU)
!ollama ps

In [ ]:
# 8. Download the results to your Mac (they are also in Google Drive)
!cd "$RESULTS" && zip -q -r /content/results.zip .
files.download("/content/results.zip")